# Olist exploratory data analysis

Reads the silver layer from HDFS with PySpark and saves one PNG per figure into
`reports/figures/` for the report.

Run it with `.\tasks.ps1 notebook` while the HDFS profile is up.

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from pipeline.common import io
from pipeline.common.spark import get_spark

INK, BLUE, MUTED, BORDER = "#16213E", "#1F5FAD", "#5B6577", "#DDE3EA"
DELAY_SCALE = ["#1E8A5A", "#D9A21B", "#D9731B", "#C2451E", "#8F1D14", "#8A94A6"]

FIGURES = Path("/workspace/reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "axes.edgecolor": BORDER,
    "axes.labelcolor": MUTED,
    "axes.titlecolor": INK,
    "axes.titlesize": 13,
    "axes.grid": True,
    "grid.color": BORDER,
    "grid.linewidth": 0.6,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "figure.dpi": 130,
})


def save(fig, name):
    """Save a figure into reports/figures and report where it went."""
    fig.tight_layout()
    path = FIGURES / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path.name}")


spark = get_spark("olist-eda")
orders = io.read_silver(spark, "orders_fact").cache()
items = io.read_silver(spark, "items_fact").cache()
customers = io.read_silver(spark, "customers")
print(f"orders_fact {orders.count():,}   items_fact {items.count():,}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/17 07:33:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


07:33:39  INFO    pipeline.spark  SparkSession ready: olist-eda on local[2] (shuffle partitions 8)


07:33:39  INFO    pipeline.io  Reading silver orders_fact         from hdfs://namenode:8020/olist/silver/orders_fact


07:33:43  INFO    pipeline.io  Reading silver items_fact          from hdfs://namenode:8020/olist/silver/items_fact


07:33:43  INFO    pipeline.io  Reading silver customers           from hdfs://namenode:8020/olist/silver/customers


orders_fact 99,441   items_fact 112,650


## 1. Orders per month

Shows why the analysis window is 2017-01 to 2018-08: 2016 is sparse and
September 2018 onwards is incomplete.

In [2]:
monthly = (
    orders.groupBy("year_month").count().orderBy("year_month").toPandas()
)

fig, ax = plt.subplots()
inside = monthly["year_month"].between("2017-01", "2018-08")
ax.bar(monthly["year_month"], monthly["count"],
       color=[BLUE if keep else "#8A94A6" for keep in inside])
ax.set_title("Orders per month (grey months fall outside the analysis window)")
ax.set_ylabel("Orders")
ax.tick_params(axis="x", rotation=90, labelsize=7)
save(fig, "orders_per_month")
monthly.tail(4)

saved orders_per_month.png


,year_month,count
21,2018-07,6292
22,2018-08,6512
23,2018-09,16
24,2018-10,4


## 2. Order status counts

Almost every order is delivered; the long tail matters for the cancel rate.

In [3]:
status = (
    orders.groupBy("status").count().orderBy(F.col("count").desc()).toPandas()
)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(status["status"], status["count"], color=BLUE)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_title("Order status counts (log scale)")
ax.set_xlabel("Orders, log scale")
for y, value in enumerate(status["count"]):
    ax.text(value, y, f"  {value:,}", va="center", fontsize=8, color=MUTED)
save(fig, "order_status_counts")
status

saved order_status_counts.png


,status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 3. Items per order

Most orders hold a single item, which is what makes the primary-category
choice a small distortion.

In [4]:
per_order = (
    orders.groupBy("n_items").count().orderBy("n_items").toPandas()
)
top = per_order[per_order["n_items"].between(1, 6)]
single_share = (
    per_order.loc[per_order["n_items"] == 1, "count"].sum() / per_order["count"].sum()
)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(top["n_items"].astype(str), top["count"], color=BLUE)
ax.set_title(f"Items per order ({single_share:.1%} of orders contain a single item)")
ax.set_xlabel("Items in the order")
ax.set_ylabel("Orders")
save(fig, "items_per_order")
print(f"single-item share: {single_share:.3%}")

saved items_per_order.png
single-item share: 89.363%


## 4. Repeat customers

Only a small share of customers order more than once, which is why the
dashboard focuses on delivery, satisfaction and sellers instead of retention.

In [5]:
per_customer = customers.groupBy("customer_unique_id").count()
repeat = per_customer.filter(F.col("count") > 1).count()
unique_customers = per_customer.count()
repeat_share = repeat / unique_customers

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(["One order", "More than one"],
       [unique_customers - repeat, repeat],
       color=[BLUE, "#8A94A6"])
ax.set_title(f"Repeat customers are rare ({repeat_share:.1%} order more than once)")
ax.set_ylabel("Distinct customers")
for x, value in enumerate([unique_customers - repeat, repeat]):
    ax.text(x, value, f"{value:,}", ha="center", va="bottom", fontsize=9, color=INK)
save(fig, "repeat_customers")
print(f"{unique_customers:,} distinct customers, {repeat:,} repeat ({repeat_share:.2%})")

saved repeat_customers.png
96,096 distinct customers, 2,997 repeat (3.12%)


## 5. Late rate by state

States far from the São Paulo seller base run noticeably later.

In [6]:
by_state = (
    orders.filter(F.col("is_delivered"))
    .groupBy("customer_state")
    .agg(
        F.count(F.lit(1)).alias("delivered"),
        F.sum(F.col("is_late").cast("int")).alias("late"),
    )
    .filter(F.col("delivered") >= 100)
    .withColumn("late_rate", F.col("late") / F.col("delivered"))
    .orderBy(F.col("late_rate").desc())
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(by_state["customer_state"], by_state["late_rate"], color=BLUE)
ax.invert_yaxis()
ax.set_title("Late rate by customer state (states with 100+ delivered orders)")
ax.set_xlabel("Share of delivered orders arriving late")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
save(fig, "late_rate_by_state")
by_state.head(5)

saved late_rate_by_state.png


,customer_state,delivered,late,late_rate
0,AL,397,85,0.214106
1,MA,717,125,0.174338
2,SE,335,51,0.152239
3,PI,476,66,0.138655
4,CE,1279,176,0.137608


## 6. Review score distribution

In [7]:
scores = (
    orders.filter(F.col("review_score").isNotNull())
    .groupBy("review_score").count().orderBy("review_score").toPandas()
)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(scores["review_score"].astype(str), scores["count"],
       color=["#8F1D14", "#C2451E", "#D9A21B", "#5B9E6F", "#1E8A5A"])
ax.set_title("Review score distribution is strongly bimodal")
ax.set_xlabel("Review score")
ax.set_ylabel("Reviewed orders")
save(fig, "review_score_distribution")
scores

saved review_score_distribution.png


,review_score,count
0,1,11363
1,2,3131
2,3,8133
3,4,19038
4,5,57008


## 7. Review score by delay bucket

The project's headline finding: satisfaction collapses as delay grows.

In [8]:
ORDER = ["On time", "1–3 days late", "4–7 days late",
         "8–14 days late", "15+ days late", "Not delivered"]

by_delay = (
    orders.filter(F.col("review_score").isNotNull())
    .groupBy("delay_bucket")
    .agg(F.avg("review_score").alias("avg_review"),
         F.count(F.lit(1)).alias("reviews"))
    .toPandas()
)
by_delay["_order"] = by_delay["delay_bucket"].apply(ORDER.index)
by_delay = by_delay.sort_values("_order")

fig, ax = plt.subplots(figsize=(9, 4.2))
bars = ax.bar(by_delay["delay_bucket"], by_delay["avg_review"], color=DELAY_SCALE)
ax.set_ylim(0, 5.4)
ax.set_title("Review scores drop sharply once an order is late")
ax.set_ylabel("Average review score")
ax.tick_params(axis="x", labelsize=8)
for bar, value in zip(bars, by_delay["avg_review"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.08,
            f"{value:.2f}", ha="center", fontsize=10, color=INK)
save(fig, "review_by_delay_bucket")
by_delay[["delay_bucket", "avg_review", "reviews"]]

saved review_by_delay_bucket.png


,delay_bucket,avg_review,reviews
1,On time,4.290386,89443
2,1–3 days late,3.291037,1852
3,4–7 days late,2.102975,1748
5,8–14 days late,1.670816,1446
0,15+ days late,1.723596,1335
4,Not delivered,1.749035,2849


In [9]:
orders.unpersist()
items.unpersist()
spark.stop()
print("figures written to reports/figures:")
for path in sorted(FIGURES.glob("*.png")):
    print(f"  {path.name}  ({path.stat().st_size // 1024} KB)")

figures written to reports/figures:
  items_per_order.png  (26 KB)
  late_rate_by_state.png  (53 KB)
  order_status_counts.png  (38 KB)
  orders_per_month.png  (50 KB)
  repeat_customers.png  (28 KB)
  review_by_delay_bucket.png  (38 KB)
  review_score_distribution.png  (26 KB)
